# Model Comparison

This notebook compares the two baselines and the three tree models on train, validation, and test. This is the first time the test set is used for any tree model.

## Planned Steps

- Load the encoded train, validation, and test sets
- Recreate the two baselines from Section 6
- Fit the base tree, the best pre-pruned tree, and the final post-pruned tree
- Print a comparison table with train F1, val F1, test F1, test accuracy, depth, and leaves

In [1]:
from itertools import product
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.tree import DecisionTreeClassifier

TRAIN_PATH = Path('data/processed/encoded_train.csv')
VAL_PATH = Path('data/processed/encoded_validation.csv')
TEST_PATH = Path('data/processed/encoded_test.csv')

if not TRAIN_PATH.exists():
    TRAIN_PATH = Path('../data/processed/encoded_train.csv')
    VAL_PATH = Path('../data/processed/encoded_validation.csv')
    TEST_PATH = Path('../data/processed/encoded_test.csv')

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

X_train = train_df.drop(columns=['Approval'])
y_train = train_df['Approval']
X_val = val_df.drop(columns=['Approval'])
y_val = val_df['Approval']
X_test = test_df.drop(columns=['Approval'])
y_test = test_df['Approval']

def metrics_row(name, y_train, train_pred, y_val, val_pred, y_test, test_pred, depth=None, leaves=None):
    return {
        'model': name,
        'train_f1': f1_score(y_train, train_pred, pos_label='Yes'),
        'val_f1': f1_score(y_val, val_pred, pos_label='Yes'),
        'test_f1': f1_score(y_test, test_pred, pos_label='Yes'),
        'test_accuracy': accuracy_score(y_test, test_pred),
        'depth': depth,
        'leaves': leaves,
    }

rows = []

always_yes_train = pd.Series(['Yes'] * len(y_train), index=y_train.index)
always_yes_val = pd.Series(['Yes'] * len(y_val), index=y_val.index)
always_yes_test = pd.Series(['Yes'] * len(y_test), index=y_test.index)
rows.append(metrics_row('Always Yes', y_train, always_yes_train, y_val, always_yes_val, y_test, always_yes_test))

threshold_train = pd.Series(['Yes' if s >= 680 else 'No' for s in train_df['CreditScore']], index=train_df.index)
threshold_val = pd.Series(['Yes' if s >= 680 else 'No' for s in val_df['CreditScore']], index=val_df.index)
threshold_test = pd.Series(['Yes' if s >= 680 else 'No' for s in test_df['CreditScore']], index=test_df.index)
rows.append(metrics_row('CreditScore >= 680', y_train, threshold_train, y_val, threshold_val, y_test, threshold_test))

base_tree = DecisionTreeClassifier(random_state=42)
base_tree.fit(X_train, y_train)
rows.append(metrics_row(
    'Base tree',
    y_train, base_tree.predict(X_train),
    y_val, base_tree.predict(X_val),
    y_test, base_tree.predict(X_test),
    depth=base_tree.get_depth(),
    leaves=base_tree.get_n_leaves(),
))

max_depth_values = [3, 5, 7, 10]
min_samples_split_values = [20, 50, 100]
min_samples_leaf_values = [10, 25, 50]
grid_rows = []
for max_depth, min_samples_split, min_samples_leaf in product(
    max_depth_values, min_samples_split_values, min_samples_leaf_values
):
    model = DecisionTreeClassifier(
        random_state=42,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
    )
    model.fit(X_train, y_train)
    grid_rows.append({
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf,
        'val_f1': f1_score(y_val, model.predict(X_val), pos_label='Yes'),
    })
grid_df = pd.DataFrame(grid_rows).sort_values(
    by=['val_f1', 'max_depth', 'min_samples_split', 'min_samples_leaf'],
    ascending=[False, True, True, True],
).reset_index(drop=True)
best_params = grid_df.iloc[0]
best_prepruned = DecisionTreeClassifier(
    random_state=42,
    max_depth=int(best_params['max_depth']),
    min_samples_split=int(best_params['min_samples_split']),
    min_samples_leaf=int(best_params['min_samples_leaf']),
)
best_prepruned.fit(X_train, y_train)
rows.append(metrics_row(
    'Best pre-pruned',
    y_train, best_prepruned.predict(X_train),
    y_val, best_prepruned.predict(X_val),
    y_test, best_prepruned.predict(X_test),
    depth=best_prepruned.get_depth(),
    leaves=best_prepruned.get_n_leaves(),
))

final_postpruned = DecisionTreeClassifier(random_state=42, ccp_alpha=0.000890)
final_postpruned.fit(X_train, y_train)
rows.append(metrics_row(
    'Final post-pruned',
    y_train, final_postpruned.predict(X_train),
    y_val, final_postpruned.predict(X_val),
    y_test, final_postpruned.predict(X_test),
    depth=final_postpruned.get_depth(),
    leaves=final_postpruned.get_n_leaves(),
))

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))


             model  train_f1   val_f1  test_f1  test_accuracy  depth  leaves
        Always Yes  0.717949 0.717949 0.717949       0.560000    NaN     NaN
CreditScore >= 680  0.535186 0.521552 0.515642       0.625833    NaN     NaN
         Base tree  1.000000 0.848665 0.852679       0.835000   23.0   518.0
   Best pre-pruned  0.904131 0.896797 0.887931       0.870000   10.0    57.0
 Final post-pruned  0.903624 0.882526 0.890522       0.875833    9.0    27.0
